# SegFormer evaluation — controlled, cross-experiment

Stand-alone evaluator for `bundle.pt` checkpoints saved by `segformer_train.ipynb`.

**What it does**
1. Builds a fixed canonical val set from `sam_coco.json` (rendered once).
2. Loads each `bundle.pt` you list and evaluates it on **the same** masks.
3. Generates an HTML report per model identical in shape to Cell 9b of the training notebook.
4. If you list >1 model, also produces a side-by-side comparison HTML.

**Constraints (current version)**
- All listed models must have been trained with `KEEP_TOP_N_CLASSES = None` (full class list).
- All compared models must share the same class taxonomy (label merge on/off must match).
- Validation set = **all** images present in `sam_coco.json` (≈450). Change `VAL_FRACTION`/`VAL_SEED` below if you want a stricter held-out subset.

**How to use**
1. Mount Drive (Cell 1).
2. Edit Cell 2 to point at your `coral_eval.py`, `sam_coco.json`, images dir, and bundles.
3. Run all cells. Reports land next to each `bundle.pt` and one consolidated copy lands in `DRIVE_DIR`.

In [6]:
# CELL 1 — environment setup
!pip install -q albumentations transformers timm pycocotools

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
# CELL 2 — CONFIG  (edit this)
import os

DRIVE_DIR        = '/content/drive/MyDrive/coral_training'

# Path to coral_eval.py — copy SegFormer_Model/coral_eval.py to Drive once,
# OR clone the repo in Colab and point at the SegFormer_Model folder.
CORAL_EVAL_DIR   = f'{DRIVE_DIR}/code'                           # contains coral_eval.py

IMAGES_DIR       = f'{DRIVE_DIR}/images'
SAM_COCO_PATH    = f'{DRIVE_DIR}/sam_coco.json'                  # step-3 SAM masks
CSV_PATH         = f'{DRIVE_DIR}/annotations_coralnet.csv'       # for class taxonomy + point accuracy

# List of bundle.pt (or best.pt) paths to evaluate. One model = single report.
# Multiple models = single reports + side-by-side comparison report.
BUNDLES = [
    f'{DRIVE_DIR}/segformer_runs/experiments/gold_masks_all_clean_14ep_20260513-1233/bundle.pt',
    # f'{DRIVE_DIR}/segformer_runs/experiments/gold_masks_all_clean_14ep_20260513-1233/bundle.pt',
]

OUTPUT_DIR       = f'{DRIVE_DIR}/eval_reports'                   # where consolidated HTMLs land

# Validation subset of sam_coco.json:
#   1.0  → use all images (default — ≈450 images)
#   0.1  → use a 10 % seeded subset (matches segformer_train's val split when SEED=42)
VAL_FRACTION     = 1.0
VAL_SEED         = 42

USE_LABEL_MERGE  = True              # must match how the bundles were trained
BATCH_SIZE       = 4
USE_AMP          = True
DO_POINT_ACC     = True              # CoralNet-equivalent point accuracy

os.makedirs(OUTPUT_DIR, exist_ok=True)

import importlib
importlib.reload(coral_eval)

print('config loaded')

config loaded


In [16]:
# CELL 3 — import the eval module + check dependencies
import sys, importlib, torch
if CORAL_EVAL_DIR not in sys.path:
    sys.path.insert(0, CORAL_EVAL_DIR)
import coral_eval; importlib.reload(coral_eval)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
if device.type != 'cuda':
    print('  ⚠ no GPU — eval will be slow. Runtime → Change runtime type → T4/L4.')

for p in [SAM_COCO_PATH, IMAGES_DIR]:
    assert os.path.exists(p), f'missing: {p}'
for b in BUNDLES:
    assert os.path.exists(b), f'missing bundle: {b}'
print(f'{len(BUNDLES)} bundle(s) ready to evaluate')

device: cuda
1 bundle(s) ready to evaluate


In [17]:
# CELL 4 — derive canonical class list from CSV
# Same taxonomy logic as segformer_train.ipynb Cell 4 with KEEP_TOP_N_CLASSES=None.
import pandas as pd, numpy as np

_EXCL = {'Unknown', 'Unkn', 'Unk', 'Off', '?', 'NA', 'nan', 'None', ''}

df = pd.read_csv(CSV_PATH, low_memory=False)
label_col = 'Label code' if 'Label code' in df.columns else 'Label'
s = df[label_col].astype(str)
s = s[~s.str.strip().str.lower().isin({c.lower() for c in _EXCL})]
if USE_LABEL_MERGE:
    s = s.map(coral_eval.apply_label_merge)
    s = s[s.notna()]
CLASSES = sorted(s.value_counts().index.tolist())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
print(f'{len(CLASSES)} classes — first 8: {CLASSES[:8]}')

87 classes — first 8: ['Aca', 'Acr', 'Agl', 'Alv', 'Anemone', 'Astrea', 'Astreo', 'BA']


In [10]:
# CELL 5 — render the canonical val set from sam_coco.json (cached on local SSD)
import json, hashlib
MASKS_CACHE = '/content/eval_masks_cache/sam_canonical'
os.makedirs(MASKS_CACHE, exist_ok=True)

_st = os.stat(SAM_COCO_PATH)
_fp = hashlib.md5(
    f'{_st.st_mtime}|{_st.st_size}|{len(CLASSES)}|merge={USE_LABEL_MERGE}'.encode()
).hexdigest()
_marker = os.path.join(MASKS_CACHE, '.fp')
_cache_ok = os.path.exists(_marker) and open(_marker).read().strip() == _fp

if _cache_ok and any(f.endswith('.png') for f in os.listdir(MASKS_CACHE)):
    print(f' cache hit → {MASKS_CACHE}')
    coco = coral_eval.preprocess_roboflow_coco(SAM_COCO_PATH, use_label_merge=USE_LABEL_MERGE)
    img_index = coral_eval.build_image_index(IMAGES_DIR)
    rendered = []
    for im in coco['images']:
        ip = coral_eval.resolve_image(im['file_name'], IMAGES_DIR, img_index)
        mp = os.path.join(MASKS_CACHE, os.path.splitext(os.path.basename(ip or ''))[0] + '.png') if ip else None
        if ip and mp and os.path.exists(mp):
            rendered.append((ip, mp))
else:
    print(f' rendering SAM masks → {MASKS_CACHE}')
    coco = coral_eval.preprocess_roboflow_coco(SAM_COCO_PATH, use_label_merge=USE_LABEL_MERGE)
    rendered = coral_eval.render_coco_to_png(coco, IMAGES_DIR, MASKS_CACHE, CLASS_TO_IDX)
    with open(_marker, 'w') as f: f.write(_fp)

# Optional sub-sampling (default keeps all)
rendered = sorted(rendered, key=lambda x: x[0])
if VAL_FRACTION < 1.0:
    rng = np.random.default_rng(VAL_SEED)
    n = max(1, int(len(rendered) * VAL_FRACTION))
    idx = sorted(rng.permutation(len(rendered))[:n].tolist())
    rendered = [rendered[i] for i in idx]

VAL_IMAGES = [r[0] for r in rendered]
VAL_MASKS  = [r[1] for r in rendered]
print(f'val set: {len(VAL_IMAGES)} images, {len(CLASSES)} classes')

EVAL_META = {
    'eval_source':    f'SAM masks ({os.path.basename(SAM_COCO_PATH)}, '
                       f'{"all imgs" if VAL_FRACTION>=1.0 else f"{int(VAL_FRACTION*100)}% subset, seed={VAL_SEED}"})',
    'n_val_images':   len(VAL_IMAGES),
    'n_classes':      len(CLASSES),
    'use_label_merge': USE_LABEL_MERGE,
}

 cache hit → /content/eval_masks_cache/sam_canonical
val set: 4432 images, 87 classes


In [20]:
# CELL 6 — evaluate every bundle
import shutil
results = []
for bp in BUNDLES:
    res = coral_eval.evaluate_bundle(
        bp, VAL_IMAGES, VAL_MASKS, CLASSES, CSV_PATH, IMAGES_DIR, device,
        batch_size=BATCH_SIZE, use_amp=USE_AMP, do_point_acc=DO_POINT_ACC,
    )
    res['bundle_path'] = bp
    results.append(res)
print(f'\n{len(results)} model(s) evaluated')


→ Loading /content/drive/MyDrive/coral_training/segformer_runs/experiments/gold_masks_all_clean_14ep_20260513-1233/bundle.pt


RuntimeError: no head_a.classifier.weight in checkpoint

In [ ]:
# CELL 7 — per-model HTML reports
for r in results:
    html = coral_eval.build_single_report_html(
        experiment_name=r['experiment_name'], classes=r['classes'], metrics=r['metrics'],
        conf=r['conf'], top_confusions=r['top_confusions'], point_acc=r['point_acc'],
        eval_meta=EVAL_META, model_meta=r['model_meta'],
    )
    # next to bundle.pt
    side = os.path.join(os.path.dirname(r['bundle_path']), 'eval_report.html')
    with open(side, 'w', encoding='utf-8') as f: f.write(html)
    # consolidated copy
    out = os.path.join(OUTPUT_DIR, f"report_{r['experiment_name']}.html")
    with open(out, 'w', encoding='utf-8') as f: f.write(html)
    print(f"  {r['experiment_name']:<40} → {out}")

In [ ]:
# CELL 8 — side-by-side comparison (only if >1 model)
if len(results) > 1:
    cmp_html = coral_eval.build_comparison_html(results, EVAL_META)
    cmp_path = os.path.join(OUTPUT_DIR, 'comparison.html')
    with open(cmp_path, 'w', encoding='utf-8') as f: f.write(cmp_html)
    print(f' comparison → {cmp_path}')
else:
    print('only one model — skipping comparison report')

In [ ]:
# CELL 9 — quick text summary in notebook
for r in results:
    s = r['metrics']['summary']
    pa = r['point_acc']['overall'] if r['point_acc'] else None
    pa_txt = f'  point_acc={pa*100:.2f}%' if pa is not None else ''
    print(f"  {r['experiment_name']:<40}  mIoU={s['mIoU']*100:5.2f}%  "
          f"FWIoU={s['FWIoU']*100:5.2f}%  pixAcc={s['pixel_accuracy']*100:5.2f}%{pa_txt}")